##Load and Filter the Data

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

gdf = gpd.read_file("/content/drive/MyDrive/AI for Safer Roads 2026 - Dataset/ADB_Innovation_Maharashtra.geojson")

# Drop ADB-flagged unreliable segments
gdf = gdf[gdf["ExcludeFromSpeedSPI"] == False].copy()

# Filter to segments where all four speed columns are present
speed_cols = ["SpeedLimit", "F85thPercentileSpeed", "MedianSpeed", "PercentOverLimit"]
gdf_valid = gdf.dropna(subset=speed_cols).copy()

print(f"Total segments: {len(gdf)}")
print(f"Valid segments for SSS: {len(gdf_valid)}")
print(gdf_valid[speed_cols].describe())

Total segments: 3577
Valid segments for SSS: 3577
       F85thPercentileSpeed  MedianSpeed  PercentOverLimit
count           3577.000000  3577.000000       3577.000000
mean              56.366404    44.664339          0.278368
std               19.064529    16.752659          0.301416
min               13.333333    12.000000          0.000000
25%               42.000000    32.000000          0.000000
50%               54.000000    42.000000          0.157658
75%               69.000000    55.666667          0.500225
max              114.666667    96.666667          1.000000


##Compute the Three SSS Components

Each component is normalised to 0–1 using percentile rank normalisation (robust to outliers, no distortion from extreme values).

Component A: Speed Delta — how far the 85th percentile speed exceeds the posted limit. This is the structural signal: if the road routinely pushes drivers 20+ km/h over the limit, the limit or the road design is wrong.

Component B: Compliance Failure — the percentage of vehicles over the limit. This captures breadth of the problem. A road where 80% of drivers speed is different from one where 15% do, even if both have the same V85.

Component C: Speed Variance — the gap between 85th percentile and median speed. High variance means the road has a mix of very fast and slower drivers sharing the same space, which is an independent crash risk factor beyond the raw speed level.

In [ ]:
# Component A: Speed Delta (V85 - posted limit)
# Ensure relevant columns are numeric
gdf_valid["F85thPercentileSpeed"] = pd.to_numeric(gdf_valid["F85thPercentileSpeed"], errors='coerce')
gdf_valid["SpeedLimit"] = pd.to_numeric(gdf_valid["SpeedLimit"], errors='coerce')
gdf_valid["MedianSpeed"] = pd.to_numeric(gdf_valid["MedianSpeed"], errors='coerce')
gdf_valid["PercentOverLimit"] = pd.to_numeric(gdf_valid["PercentOverLimit"], errors='coerce')

# Drop rows where these key numeric conversions resulted in NaN
gdf_valid.dropna(subset=["F85thPercentileSpeed", "SpeedLimit", "MedianSpeed", "PercentOverLimit"], inplace=True)

# Clip at 0 — negative deltas (traffic below limit) are treated as zero risk contribution
gdf_valid["speed_delta"] = (gdf_valid["F85thPercentileSpeed"] - gdf_valid["SpeedLimit"]).clip(lower=0)

# Component B: Compliance failure — % of vehicles over the limit
# Already a 0-100 value in the dataset
gdf_valid["compliance_failure"] = gdf_valid["PercentOverLimit"]

# Component C: Speed variance — gap between V85 and median
# Captures how spread out the speed distribution is
gdf_valid["speed_variance"] = (gdf_valid["F85thPercentileSpeed"] - gdf_valid["MedianSpeed"]).clip(lower=0)

# Percentile-rank normalise each to 0-1
def pct_rank(s):
    if s.nunique() <= 1:
        return pd.Series(0.0, index=s.index)
    return s.rank(pct=True)

gdf_valid["delta_norm"]     = pct_rank(gdf_valid["speed_delta"])
gdf_valid["compliance_norm"] = pct_rank(gdf_valid["compliance_failure"])
gdf_valid["variance_norm"]  = pct_rank(gdf_valid["speed_variance"])

##Compute the SSS and Assign Tiers

Weights: speed delta gets the most weight because it's the strongest structural signal.

Compliance failure is second because breadth of speeding reflects enforcement and limit realism.

Variance is third because it adds independent risk information beyond the other two.

Each road segment now has an SSS (0–100) and an SSS_Tier (Critical / High / Moderate / Low) derived entirely from speed behaviour — nothing circular, nothing that uses the structural features you'll feed the model later. The score is transparent: a ministry official can be told "this road scores 82 because 78% of drivers exceed the limit and the typical fast driver does 25 km/h over — that's Critical."

In [ ]:
# Weights — document these in your methodology section
W_DELTA      = 0.50
W_COMPLIANCE = 0.35
W_VARIANCE   = 0.15

assert abs(W_DELTA + W_COMPLIANCE + W_VARIANCE - 1.0) < 1e-9

gdf_valid["SSS_raw"] = (
    W_DELTA      * gdf_valid["delta_norm"] +
    W_COMPLIANCE * gdf_valid["compliance_norm"] +
    W_VARIANCE   * gdf_valid["variance_norm"]
)

# Scale to 0-100
gdf_valid["SSS"] = (gdf_valid["SSS_raw"] * 100).round(1)

# Safe System penalty floor
# WHO thresholds: urban roads > 50 km/h, rural undivided > 80 km/h are structurally unsafe
# If a segment's posted limit exceeds its Safe System threshold, floor the score at 60

SAFE_SYSTEM_LIMITS = {
    "motorway":  110,
    "trunk":      80,
    "primary":    80,
    "secondary":  50,
}

def safe_system_floor(row):
    road_class = str(row.get("RoadClass", "")).lower()
    limit = row.get("SpeedLimit", np.nan)
    land_use = str(row.get("LandUse", "")).lower()

    # Urban roads: Safe System cap is 50 km/h
    if "urban" in land_use and not pd.isna(limit) and limit > 50:
        return max(row["SSS"], 60)

    # Class-specific thresholds for rural
    threshold = SAFE_SYSTEM_LIMITS.get(road_class, None)
    if threshold and not pd.isna(limit) and limit > threshold:
        return max(row["SSS"], 60)

    return row["SSS"]

gdf_valid["SSS"] = gdf_valid.apply(safe_system_floor, axis=1)

# Tier classification
def assign_tier(score):
    if score >= 80:
        return "Critical"
    elif score >= 60:
        return "High"
    elif score >= 40:
        return "Moderate"
    else:
        return "Low"

gdf_valid["SSS_Tier"] = gdf_valid["SSS"].apply(assign_tier)

# Quick summary
print(gdf_valid["SSS_Tier"].value_counts())
print(gdf_valid["SSS"].describe())

SSS_Tier
High        1987
Low          701
Critical     596
Moderate     293
Name: count, dtype: int64
count    3577.000000
mean       59.647134
std        20.097805
min        16.800000
25%        56.500000
50%        60.000000
75%        72.300000
max        98.900000
Name: SSS, dtype: float64


In [ ]:
gdf_valid.to_file("maharashtra_sss_scored.geojson", driver="GeoJSON")
print(f"Exported {len(gdf_valid)} segments to maharashtra_sss_scored.geojson")
print(gdf_valid["SSS_Tier"].value_counts())

Exported 3577 segments to maharashtra_sss_scored.geojson
SSS_Tier
High        1987
Low          701
Critical     596
Moderate     293
Name: count, dtype: int64


##Compute Sinuosity (Road Curvature) from Geometry

A perfectly straight road = 1.0. A winding road will be higher. This is your curvature proxy.

In [ ]:
from shapely.geometry import LineString

def compute_sinuosity(geom):
    if geom is None or geom.is_empty:
        return 1.0
    try:
        coords = list(geom.coords)
        if len(coords) < 2:
            return 1.0
        path_length = geom.length
        straight_line = LineString([coords[0], coords[-1]]).length
        if straight_line == 0:
            return 1.0
        return path_length / straight_line
    except:
        return 1.0

gdf_valid["sinuosity"] = gdf_valid.geometry.apply(compute_sinuosity)
print(gdf_valid["sinuosity"].describe())

count    3577.000000
mean        2.393359
std        37.371845
min         1.000000
25%         1.019910
50%         1.057454
75%         1.131793
max      2078.646132
Name: sinuosity, dtype: float64


Capping outliers

In [ ]:
cap = gdf_valid["sinuosity"].quantile(0.99)
gdf_valid["sinuosity"] = gdf_valid["sinuosity"].clip(upper=cap)
print(gdf_valid["sinuosity"].describe())

count    3577.000000
mean        1.109880
std         0.157439
min         1.000000
25%         1.019910
50%         1.057454
75%         1.131793
max         2.007514
Name: sinuosity, dtype: float64


##Merge VRU File and Prepare Model Features

Load the VRU output and merge it in on the segment ID.

In [ ]:
gdf_vru = gpd.read_file("/content/drive/MyDrive/AI for Safer Roads 2026 - Dataset/maharashtra_vru_exposure_output.geojson")

# Merge VRU_exposure_index onto your valid scored segments
gdf_model = gdf_valid.merge(
    gdf_vru[["OBJECTID", "VRU_exposure_index"]],
    on="OBJECTID",
    how="left"
)

# Fill any segments that didn't match with the median (shouldn't be many)
gdf_model["VRU_exposure_index"] = gdf_model["VRU_exposure_index"].fillna(
    gdf_model["VRU_exposure_index"].median()
)

print(f"Model dataset shape: {gdf_model.shape}")
print(gdf_model["VRU_exposure_index"].describe())

Model dataset shape: (3577, 39)
count    3577.000000
mean       47.502987
std         9.470827
min        31.497283
25%        46.619565
50%        46.619565
75%        46.619565
max        99.701087
Name: VRU_exposure_index, dtype: float64


##Encode Categorical Features as XGBoost needs numbers, not strings.

In [ ]:
# Road class encoding — higher number = more pedestrian-vehicle mixing risk
road_class_map = {
    "motorway":  1,
    "trunk":     2,
    "primary":   3,
    "secondary": 4,
}
gdf_model["road_class_encoded"] = gdf_model["RoadClass"].str.lower().map(road_class_map).fillna(3)

# Land use encoding
gdf_model["land_use_encoded"] = gdf_model["LandUse"].str.lower().map({
    "urban": 1,
    "rural": 0
}).fillna(0)

# Final feature set — ONLY structural features, no speed metrics
FEATURE_COLS = [
    "VRU_exposure_index",
    "sinuosity",
    "road_class_encoded",
    "land_use_encoded"
]

TARGET_COL = "SSS"

# Drop rows where any feature or target is missing
gdf_model = gdf_model.dropna(subset=FEATURE_COLS + [TARGET_COL])

X = gdf_model[FEATURE_COLS]
y = gdf_model[TARGET_COL]

print(f"Training on {len(X)} segments")
print(f"Target (SSS) distribution:\n{y.describe()}")

Training on 3577 segments
Target (SSS) distribution:
count    3577.000000
mean       59.647134
std        20.097805
min        16.800000
25%        56.500000
50%        60.000000
75%        72.300000
max        98.900000
Name: SSS, dtype: float64


##Train XGBoost with Cross-Validation

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=20,
    eval_metric="mae",
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_pred = model.predict(X_test)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R²:   {r2_score(y_test, y_pred):.3f}")

# Cross-validation on full dataset for robustness check
cv_scores = cross_val_score(
    xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                     subsample=0.8, colsample_bytree=0.8, random_state=42),
    X, y, cv=5, scoring="r2"
)
print(f"5-fold CV R² scores: {cv_scores.round(3)}")
print(f"Mean CV R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

MAE:  12.96
R²:   0.205
5-fold CV R² scores: [0.109 0.068 0.024 0.137 0.198]
Mean CV R²: 0.107 ± 0.060
